# Parte 1 – Scraping de decretos de emergencia

**Temporada:** 1 de enero – 31 de mayo de 2023  
**Fuente:** gob.pe – Buscador de normas de la PCM  
**Salida:** `datos/decretos_lluvias.csv` y `datos/decretos_por_departamento.csv`

## 1. Objetivo

Recopilar todos los decretos supremos que declaran o prorrogan el estado de emergencia por lluvias/precipitaciones durante la temporada 2023, enriquecerlos con su título completo, clasificarlos y contar cuántos mencionan a cada departamento del Perú.

## 2. Importación de librerías

In [1]:
import os
import re
import time
import calendar
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## 3. Configuración

Definimos las constantes del proyecto: URL base, año, meses, pausas y User-Agent.

In [2]:
# URL base de búsqueda de normas en gob.pe
BASE_URL = "https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent"

# Año de la temporada
AÑO = 2023

# Meses a scrapear (1=enero, 2=febrero, ..., 5=mayo)
MESES = [1, 2, 3, 4, 5]

# Pausa entre páginas de Selenium (segundos)
PAUSA_SELENIUM = 2

# Pausa entre pedidos de requests (segundos)
PAUSA_REQUESTS = 1

# Reintentos por página cuando la conexión falla, y timeout de cada pedido
REINTENTOS = 4
TIMEOUT_REQUESTS = 30

# User-Agent para identificarnos como navegador
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Lista de departamentos del Perú para la detección
DEPARTAMENTOS = [
    "Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
    "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
    "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
    "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
    "San Martín", "Tacna", "Tumbes", "Ucayali"
]

## 4. robots.txt

Antes de hacer scraping, revisamos el archivo `robots.txt` de gob.pe para entender qué está permitido y qué no.

In [3]:
# Descargamos el robots.txt
respuesta = requests.get("https://www.gob.pe/robots.txt", headers=HEADERS)
robots_txt = respuesta.text
print(robots_txt[:2000])  # Mostramos los primeros 2000 caracteres

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



### Explicación del robots.txt

**¿Qué prohíbe?** Para todos los bots (`User-agent: *`), gob.pe bloquea tres rutas:

- `Disallow: /admin/` — el panel de administración del sitio.
- `Disallow: /*?sheet=` y `Disallow: /*&sheet=` — cualquier URL con el parámetro `sheet`, que sirve para exportar tablas. Es una forma de evitar que los bots disparen exportaciones costosas.

Además hay reglas por bot: a `AhrefsBot` (un rastreador de SEO comercial) le prohíbe **todo el sitio** con `Disallow: /`.

**¿Está permitida la página de búsquedas?** Sí. `/busquedas` no aparece en ninguna regla `Disallow`, ni en las generales ni en las específicas por bot. Nuestra ruta está permitida.

**¿Por qué igual conviene hacer pausas?** Porque el propio archivo nos dice el ritmo que gob.pe considera aceptable. A los bots de IA les pide explícitamente:

- `GPTBot` → `Crawl-delay: 5` (5 segundos entre pedidos)
- `OAI-SearchBot` → `Crawl-delay: 2` (2 segundos entre pedidos)

No hay un `Crawl-delay` para `User-agent: *`, así que a nosotros no nos aplica ninguno de forma obligatoria. Pero la ausencia de una regla no es un permiso para golpear el servidor: si a un rastreador comercial le piden entre 2 y 5 segundos, ese es el ritmo que el sitio tolera. Por eso usamos **2 segundos entre páginas de búsqueda** (`PAUSA_SELENIUM`) y **1 segundo entre páginas de norma** (`PAUSA_REQUESTS`), que son unos 50 pedidos livianos.

Las pausas además nos convienen a nosotros: sin ellas, gob.pe empieza a cortar conexiones por timeout, y un pedido que falla significa un decreto que desaparece del análisis.

## 5. Scraping mensual con Selenium

La página de gob.pe carga los resultados con JavaScript, por lo que necesitamos Selenium. Vamos a buscar mes por mes para poder verificar los conteos.

In [4]:
def obtener_rango_fechas(mes, año):
    """
    Devuelve las fechas de inicio y fin de un mes en formato DD-MM-AAAA.
    """
    ultimo_dia = calendar.monthrange(año, mes)[1]
    return f"01-{mes:02d}-{año}", f"{ultimo_dia}-{mes:02d}-{año}"


def obtener_enlaces_mes(driver, mes, año):
    """
    Abre la búsqueda de un mes y extrae los datos de cada resultado.
    Devuelve una lista de diccionarios y el total de resultados que muestra la página.
    """
    desde, hasta = obtener_rango_fechas(mes, año)
    url = f"{BASE_URL}&desde={desde}&hasta={hasta}"

    driver.get(url)

    # Esperamos a que carguen los resultados (la página usa JavaScript)
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "article"))
    )

    # Leemos el número total de resultados que muestra la página
    total_elemento = driver.find_element(By.XPATH, "//span[contains(., 'Resultados')]")
    total = int(total_elemento.text.split()[0])

    # Encontramos todos los artículos (cada resultado es un <article>)
    articulos = driver.find_elements(By.CSS_SELECTOR, "article")

    resultados = []
    for articulo in articulos:
        try:
            # El número de la norma está en h4 a.link-principal
            enlace_principal = articulo.find_element(By.CSS_SELECTOR, "h4 a.link-principal")
            numero = enlace_principal.text.strip()
            url_norma = enlace_principal.get_attribute("href")

            # La fecha está en span con clase 'text-gray-900'
            fecha_elemento = articulo.find_element(By.CSS_SELECTOR, "span[class*='text-gray-900']")
            fecha = fecha_elemento.text.strip()

            # El título está en p con id que contiene 'description'
            titulo_elemento = articulo.find_element(By.CSS_SELECTOR, "p[id*='description']")
            titulo = titulo_elemento.text.strip()

            resultados.append({
                "numero": numero,
                "fecha": fecha,
                "titulo": titulo,
                "enlace": url_norma
            })
        except Exception:
            # Si un artículo no tiene el formato esperado, lo saltamos
            continue

    return resultados, total

### Ejecutamos el scraping para todos los meses

Procesamos enero a mayo de 2023, con una pausa de 2 segundos entre cada página.

In [5]:
# Configuramos el driver de Chrome
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # No abrir ventana del navegador
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# Lista para guardar todos los resultados
todos_los_resultados = []
totales_por_mes = []

for mes in MESES:
    print(f"Procesando mes {mes}...")
    resultados_mes, total = obtener_enlaces_mes(driver, mes, AÑO)

    # Guardamos el total que dice la página
    totales_por_mes.append({"mes": mes, "resultados_totales": total})

    # Agregamos el mes a cada resultado
    for r in resultados_mes:
        r["mes"] = mes

    todos_los_resultados.extend(resultados_mes)
    print(f"  Encontrados: {len(resultados_mes)} de {total} totales")

    # Pausa entre páginas
    time.sleep(PAUSA_SELENIUM)

driver.quit()
print(f"\nTotal de normas scrapeadas: {len(todos_los_resultados)}")

Procesando mes 1...


  Encontrados: 11 de 11 totales


Procesando mes 2...


  Encontrados: 9 de 9 totales


Procesando mes 3...


...

Procesando mes 4...


  Encontrados: 11 de 11 totales


Procesando mes 5...


  Encontrados: 8 de 8 totales



Total de normas scrapeadas: 53


## 6. Verificación

Comparamos los resultados extraídos con los totales que muestra la página para cada mes.

In [6]:
# Creamos un DataFrame con todos los resultados
df = pd.DataFrame(todos_los_resultados)

# Contamos cuántos extraímos por mes
extraidos_por_mes = df.groupby("mes").size().reset_index(name="resultados_extraidos")

# Creamos la tabla de verificación
tabla_verificacion = pd.DataFrame(totales_por_mes)
tabla_verificacion = tabla_verificacion.merge(extraidos_por_mes, on="mes", how="left")
tabla_verificacion["resultados_extraidos"] = tabla_verificacion["resultados_extraidos"].fillna(0).astype(int)
tabla_verificacion["coincide"] = tabla_verificacion["resultados_totales"] == tabla_verificacion["resultados_extraidos"]

print(tabla_verificacion)
print()

# Si un mes no coincide, la página paginó los resultados o cambió el HTML:
# seguir produciría conteos incompletos.
if tabla_verificacion["coincide"].all():
    print("✓ Todos los meses coinciden. Podemos continuar.")
else:
    meses_malos = tabla_verificacion.loc[~tabla_verificacion["coincide"], "mes"].tolist()
    raise RuntimeError(f"Los meses {meses_malos} no coinciden: revisar paginación o selectores.")

   mes  resultados_totales  resultados_extraidos  coincide
0    1                  11                    11      True
1    2                   9                     9      True
2    3                  14                    14      True
3    4                  11                    11      True
4    5                   8                     8      True

✓ Todos los meses coinciden. Podemos continuar.


## 7. Eliminación de duplicados

Algunos decretos pueden aparecer en más de una búsqueda (por ejemplo, si se publicaron cerca de un cambio de mes). Eliminamos los repetidos.

In [7]:
# Contamos cuántos duplicados hay
duplicados = df["enlace"].duplicated().sum()
print(f"Duplicados encontrados: {duplicados}")

# Eliminamos duplicados (nos quedamos con la primera aparición)
df = df.drop_duplicates(subset="enlace").reset_index(drop=True)
print(f"Registros después de eliminar duplicados: {len(df)}")

Duplicados encontrados: 0
Registros después de eliminar duplicados: 53


## 8. Filtro PCM

Aunque la búsqueda filtra por PCM, se cuelan normas de otras instituciones (por ejemplo, resoluciones del OECE). Nos quedamos solo con las que tienen `/institucion/pcm/` en el enlace.

In [8]:
# Filtramos solo normas de PCM
df_pcm = df[df["enlace"].str.contains("/institucion/pcm/", na=False)].copy()

no_pcm = len(df) - len(df_pcm)
print(f"Normas no PCM eliminadas: {no_pcm}")
print(f"Normas PCM restantes: {len(df_pcm)}")

Normas no PCM eliminadas: 4
Normas PCM restantes: 49


## 9. Títulos completos con requests + BeautifulSoup

Los títulos en la lista aparecen cortados ('...'). Visitamos cada página de decreto para obtener el título completo.

In [9]:
def obtener_titulo_completo(url):
    """
    Visita la página de un decreto y obtiene su título completo.
    Reintenta con espera creciente: una página perdida hace desaparecer
    el decreto de todo el análisis posterior sin dejar rastro.
    """
    for intento in range(1, REINTENTOS + 1):
        try:
            respuesta = requests.get(url, headers=HEADERS, timeout=TIMEOUT_REQUESTS)
            respuesta.raise_for_status()
            sopa = BeautifulSoup(respuesta.text, "html.parser")

            # El título completo está en un div con class 'description'
            titulo_div = sopa.find("div", class_="description")
            if titulo_div is None:
                return None

            titulo = titulo_div.text.strip()
            # Limpiamos el texto extra al final (DS N° ..., PDF, Descargar, etc.)
            titulo = re.sub(r"\s*DS\s+N[°º].*$", "", titulo, flags=re.IGNORECASE | re.DOTALL)
            titulo = re.sub(r"\s+PDF\s+.*$", "", titulo, flags=re.IGNORECASE)
            titulo = re.sub(r"\s+Descargar\s*$", "", titulo)
            return titulo
        except requests.RequestException as e:
            print(f"  Intento {intento}/{REINTENTOS} falló para {url}: {e}")
            if intento < REINTENTOS:
                time.sleep(PAUSA_REQUESTS * intento)
    return None


# Obtenemos el título completo de cada decreto, con pausa entre pedidos
print("Obteniendo títulos completos...")
titulos = []
for url in df_pcm["enlace"]:
    titulos.append(obtener_titulo_completo(url))
    time.sleep(PAUSA_REQUESTS)

df_pcm["titulo_completo"] = titulos

faltantes = df_pcm[df_pcm["titulo_completo"].isna()]
print(f"Títulos obtenidos: {len(df_pcm) - len(faltantes)} de {len(df_pcm)}")

# Sin título completo no se puede clasificar ni detectar departamentos:
# continuar sesgaría los conteos en silencio.
if len(faltantes) > 0:
    print("\nNormas sin título completo:")
    for _, fila in faltantes.iterrows():
        print(f"  - {fila['numero']}: {fila['enlace']}")
    raise RuntimeError(
        f"{len(faltantes)} normas sin título completo. "
        "Volver a ejecutar esta celda antes de continuar."
    )

Obteniendo títulos completos...


Títulos obtenidos: 49 de 49


## 10. Clasificación: es_lluvia, tipo, motivo

Creamos las columnas de clasificación usando el título completo en minúsculas.

In [10]:
# Creamos una columna con el título en minúsculas para facilitar la búsqueda
df_pcm["titulo_minusculas"] = df_pcm["titulo_completo"].str.lower()

# es_lluvia: True si contiene 'lluvia' o 'precipitaciones'
df_pcm["es_lluvia"] = (
    df_pcm["titulo_minusculas"].str.contains("lluvia", na=False) |
    df_pcm["titulo_minusculas"].str.contains("precipitaciones", na=False)
)

# tipo: prorroga, declara u otro
def clasificar_tipo(titulo):
    if "prórroga" in titulo or "prorroga" in titulo:
        return "prorroga"
    if "declara" in titulo:
        return "declara"
    return "otro"

df_pcm["tipo"] = df_pcm["titulo_minusculas"].apply(clasificar_tipo)

# motivo: peligro inminente, impacto de daños u otro
def clasificar_motivo(titulo):
    if "peligro inminente" in titulo:
        return "peligro inminente"
    if "impacto de daños" in titulo:
        return "impacto de daños"
    return "otro"

df_pcm["motivo"] = df_pcm["titulo_minusculas"].apply(clasificar_motivo)

print(df_pcm["tipo"].value_counts().to_string())
print()
print(df_pcm["motivo"].value_counts().to_string())

# Revisamos las filas que quedaron como 'otro'.
# El detalle se lista sobre las normas de lluvia, que son las que llegan al CSV final:
# las demás se descartan en el paso siguiente.
for columna in ["tipo", "motivo"]:
    otros = df_pcm[df_pcm[columna] == "otro"]
    otros_lluvia = otros[otros["es_lluvia"]]
    print(f"\n=== {columna} == 'otro': {len(otros)} de {len(df_pcm)} normas PCM")
    print(f"    de las cuales son de lluvia: {len(otros_lluvia)}")
    for _, fila in otros_lluvia.iterrows():
        print(f"  - {fila['numero']}: {fila['titulo_completo']}")

tipo
declara     28
prorroga    21

motivo
otro                 28
impacto de daños     16
peligro inminente     5

=== tipo == 'otro': 0 de 49 normas PCM
    de las cuales son de lluvia: 0

=== motivo == 'otro': 28 de 49 normas PCM
    de las cuales son de lluvia: 2
  - Decreto Supremo N.° 043-2023-PCM: Decreto Supremo que declara el Estado de Emergencia Nacional, por desastre de gran magnitud, a consecuencia de intensas precipitaciones pluviales en los departamentos de Lambayeque, Piura y Tumbes
  - Decreto Supremo N.° 065-2023-PCM: Decreto Supremo que prorroga el Estado de Emergencia Nacional, por desastre de gran magnitud, a consecuencia de intensas precipitaciones pluviales en varios distritos de las provincias de los departamentos de Lambayeque, Piura y Tumbes


### Revisión de las filas clasificadas como "otro"

**`tipo == "otro"`: 0 filas.** Las 49 normas de la PCM caen en `declara` (28) o `prorroga` (21). Toda norma que declara o prorroga un estado de emergencia usa uno de esos dos verbos en el título, así que la clasificación queda completa.

**`motivo == "otro"`: 28 filas, de las cuales solo 2 son de lluvia.** Las otras 26 se descartan en el paso siguiente por no ser de lluvia: son emergencias por otras causas (carreteras de la Red Vial Nacional, corredores viales) y hasta dos Directivas de 2015 sobre simulacros del Fenómeno El Niño que se colaron en la búsqueda.

Las 2 filas de lluvia que quedaron en `"otro"` son:

- `Decreto Supremo N.° 043-2023-PCM` — *"declara el Estado de Emergencia Nacional, **por desastre de gran magnitud**, a consecuencia de intensas precipitaciones pluviales en los departamentos de Lambayeque, Piura y Tumbes"*
- `Decreto Supremo N.° 065-2023-PCM` — la prórroga de ese mismo estado de emergencia nacional, con la misma fórmula.

**¿Están bien clasificadas? Sí.** Y es un hallazgo, no un error del código: estos dos decretos usan un **tercer motivo** que la consigna no listó, *"por desastre de gran magnitud"*. Son la declaratoria de Estado de Emergencia **Nacional** por el ciclón Yaku de marzo de 2023, que golpeó el norte del país.

La diferencia es de fondo, no de redacción:

- *peligro inminente* → se declara **antes** del daño, por pronóstico;
- *impacto de daños* → se declara **después**, por daño ya ocurrido y acotado a distritos;
- *desastre de gran magnitud* → figura del SINAGERD para eventos que superan la capacidad de respuesta regional y activan el nivel nacional.

Forzar estos dos decretos a una de las dos categorías pedidas sería inventar el dato. Los dejamos en `"otro"` y lo documentamos, que es la respuesta honesta. Esto además importa para la Parte 3: al calcular el porcentaje de declaratorias por peligro inminente frente a impacto de daños, estas 2 filas no pertenecen a ninguno de los dos grupos y hay que tratarlas aparte.

## 11. Filtrar solo normas de lluvia

Nos quedamos solo con las normas que mencionan lluvia o precipitaciones.

In [11]:
# Filtramos solo normas de lluvia
df_lluvias = df_pcm[df_pcm["es_lluvia"] == True].copy()

print(f"Normas de lluvia: {len(df_lluvias)}")
print("\nPrimeras 5 filas:")
df_lluvias[["numero", "fecha", "titulo_completo", "tipo", "motivo"]].head()

Normas de lluvia: 20

Primeras 5 filas:


,numero,fecha,titulo_completo,tipo,motivo
8,Decreto Supremo N.° 008-2023-PCM,Publicado: 13 de enero de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,peligro inminente
9,Decreto Supremo N.° 007-2023-PCM,Publicado: 13 de enero de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños
13,Decreto Supremo N.° 024-2023-PCM,Publicado: 17 de febrero de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños
18,Decreto Supremo N.° 019-2023-PCM,Publicado: 6 de febrero de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños
21,Decreto Supremo N.° 044-2023-PCM,Publicado: 29 de marzo de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños


## 12. Detección de departamentos

Identificamos qué departamentos menciona cada decreto. Usamos expresiones regulares con palabra completa para evitar falsos positivos (ej: 'ica' dentro de 'huancavelica').

In [12]:
def detectar_departamentos(titulo):
    """
    Devuelve la lista de departamentos mencionados en el título, sin repetir.
    Usa palabra completa (\\b) para que 'Ica' no coincida dentro de 'Huancavelica',
    y agrega cada departamento una sola vez aunque aparezca varias veces
    (por ejemplo 'provincia de Ica del departamento de Ica').
    """
    encontrados = []
    for depto in DEPARTAMENTOS:
        if re.search(r"\b" + re.escape(depto) + r"\b", titulo):
            encontrados.append(depto)
    return encontrados


# Detectamos departamentos en cada decreto
df_lluvias["departamentos"] = df_lluvias["titulo_completo"].apply(detectar_departamentos)

# declaratorias y prorrogas son categorías disjuntas: cada decreto cae en una sola.
registros = []
for depto in DEPARTAMENTOS:
    menciona = df_lluvias["departamentos"].apply(lambda lista: depto in lista)
    registros.append({
        "departamento": depto,
        "declaratorias": int((menciona & (df_lluvias["tipo"] == "declara")).sum()),
        "prorrogas": int((menciona & (df_lluvias["tipo"] == "prorroga")).sum()),
    })

df_departamentos = pd.DataFrame(registros)

print(df_departamentos.to_string(index=False))
print()

# Control: ningún decreto debe perderse al repartirlo entre las dos columnas
suma_columnas = df_departamentos["declaratorias"].sum() + df_departamentos["prorrogas"].sum()
suma_menciones = df_lluvias["departamentos"].apply(len).sum()
print(f"Menciones departamento-decreto: {suma_menciones}")
print(f"declaratorias + prorrogas:      {suma_columnas}")
if suma_columnas != suma_menciones:
    raise RuntimeError("Hay decretos cuyo 'tipo' no es declara ni prorroga: revisar la clasificación.")

 departamento  declaratorias  prorrogas
     Amazonas              3          1
       Áncash              6          1
     Apurímac              1          0
     Arequipa              4          0
     Ayacucho              5          1
    Cajamarca              4          0
       Callao              1          0
        Cusco              2          0
 Huancavelica              3          0
      Huánuco              4          0
          Ica              3          0
        Junín              2          0
  La Libertad              5          0
   Lambayeque              3          1
         Lima              6          0
       Loreto              1          0
Madre de Dios              0          0
     Moquegua              2          0
        Pasco              2          0
        Piura              3          1
         Puno              1          0
   San Martín              4          0
        Tacna              2          0
       Tumbes              2          1


In [13]:
# Comprobación de los dos riesgos del paso 12, con decretos reales de la temporada.

# Riesgo 1: doble conteo. Un decreto que nombra Ica como provincia Y como departamento.
caso_ica = df_lluvias[df_lluvias["titulo_completo"].str.contains("del departamento de Ica", na=False)]
for _, fila in caso_ica.iterrows():
    print(f"{fila['numero']}")
    print(f"  Título: {fila['titulo_completo']}")
    print(f"  Veces que aparece 'Ica' como palabra: {len(re.findall(r'\bIca\b', fila['titulo_completo']))}")
    print(f"  Departamentos detectados: {fila['departamentos']}")
    print(f"  ¿'Ica' una sola vez en la lista? {fila['departamentos'].count('Ica') == 1}")
    print()

# Riesgo 2: falso positivo. 'ica' vive dentro de 'Huancavelica'.
prueba = "Decreto Supremo que declara el Estado de Emergencia en el departamento de Huancavelica"
print(f"Sin \\b  -> {'ica' in prueba.lower()}  (falso positivo: contaría Ica)")
print(f"Con \\b  -> {detectar_departamentos(prueba)}  (correcto: solo Huancavelica)")

Decreto Supremo N.° 044-2023-PCM
  Título: Decreto Supremo que declara el Estado de Emergencia en algunos distritos de las provincias de Leoncio Prado y Marañón del departamento de Huánuco; y, en algunos distritos de las provincias de Nasca, Ica y Palpa del departamento de Ica, por impacto de daños a consecuencia de intensas precipitaciones pluviales
  Veces que aparece 'Ica' como palabra: 2
  Departamentos detectados: ['Huánuco', 'Ica']
  ¿'Ica' una sola vez en la lista? True

Sin \b  -> True  (falso positivo: contaría Ica)
Con \b  -> ['Huancavelica']  (correcto: solo Huancavelica)


### Cómo evitamos contar mal los departamentos

Hay dos formas distintas de equivocarse acá, y cada una necesita su propia defensa.

**Riesgo 1 — falso positivo: `"ica"` vive dentro de `"Huancavelica"`.**

Si buscamos el texto suelto en el título en minúsculas, `"ica" in "huancavelica"` es `True`, y un decreto que solo menciona Huancavelica contaría también para Ica. La defensa es doble: buscamos en el **título original** (con mayúsculas) y con `\b` (*word boundary*), que exige que la coincidencia empiece y termine en un límite de palabra:

```python
re.search(r"\b" + re.escape(depto) + r"\b", titulo)
```

La celda anterior lo comprueba: sobre `"...en el departamento de Huancavelica"`, la búsqueda suelta devuelve `True` para Ica, y la nuestra devuelve solo `['Huancavelica']`.

**Riesgo 2 — doble conteo: "Ica" también es una provincia.**

El caso real de nuestra temporada es el **`Decreto Supremo N.° 044-2023-PCM`**:

> *"...declara el Estado de Emergencia en algunos distritos de las provincias de Leoncio Prado y Marañón del departamento de Huánuco; y, en algunos distritos de las provincias de Nasca, **Ica** y Palpa del **departamento de Ica**, por impacto de daños a consecuencia de intensas precipitaciones pluviales"*

La palabra `Ica` aparece **dos veces**: una como provincia y otra como departamento. Si contáramos ocurrencias, este decreto le sumaría 2 a Ica siendo uno solo.

No pasa, porque `detectar_departamentos` pregunta por **presencia, no por cantidad**: `re.search` devuelve la primera coincidencia y `encontrados.append(depto)` corre una sola vez por departamento, dentro de un `for` que recorre la lista de los 25. La lista que devuelve no puede tener repetidos. La celda anterior lo verifica sobre este decreto: 2 apariciones de la palabra, `Ica` una sola vez en la lista.

**Limitación que conviene declarar.** Solo detectamos departamentos **nombrados explícitamente** en el título. Un decreto que dijera únicamente *"la provincia de Nasca"* sin agregar *"del departamento de Ica"* no se contaría, porque no cruzamos provincias ni distritos contra un padrón de ubigeos. Por eso el conteo es un piso, no un total exacto.

### Criterio para `declaratorias` y `prorrogas`

La consigna pide las columnas `departamento`, `declaratorias` y `prorrogas`, pero no dice si se solapan. Las tratamos como **categorías disjuntas**:

- `declaratorias` → decretos con `tipo == "declara"`
- `prorrogas` → decretos con `tipo == "prorroga"`

Así las dos columnas se pueden sumar sin doble conteo, y esa suma es el total de decretos que mencionan al departamento. Como en nuestra temporada `tipo == "otro"` tiene 0 filas, no se pierde ninguna norma en el reparto: la celda anterior lo comprueba contrastando las 78 menciones departamento-decreto contra la suma de las dos columnas.

La alternativa era que `declaratorias` fuera el total y `prorrogas` un subconjunto suyo, pero entonces una columna estaría contenida en la otra sin que ningún nombre lo indique, y quien lea el CSV sumaría mal.

## 13. Guardado de archivos

Guardamos los dos CSV finales con encoding `utf-8-sig` para que las tildes se vean bien en Excel.

In [14]:
# Creamos la carpeta datos si no existe
os.makedirs("datos", exist_ok=True)

# CSV 1: decretos_lluvias.csv
df_lluvias_final = df_lluvias[["numero", "fecha", "titulo_completo", "tipo", "motivo", "enlace"]].copy()
df_lluvias_final.columns = ["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]
df_lluvias_final.to_csv("datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")
print(f"Guardado: datos/decretos_lluvias.csv ({len(df_lluvias_final)} filas)")

# CSV 2: decretos_por_departamento.csv
df_departamentos.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
print(f"Guardado: datos/decretos_por_departamento.csv ({len(df_departamentos)} filas)")

print("\n✓ Parte 1 completada exitosamente.")

Guardado: datos/decretos_lluvias.csv (20 filas)
Guardado: datos/decretos_por_departamento.csv (25 filas)

✓ Parte 1 completada exitosamente.


## 14. Limitaciones de esta extracción

Lo que este notebook **no** garantiza, dicho explícitamente para que no se lea como más de lo que es:

1. **No hay paginación.** Leemos los `<article>` de la primera página de cada mes. En la temporada 2023 ningún mes pasa de 14 resultados, así que entra todo, y la tabla de verificación del paso 6 **lanza una excepción** si los extraídos no coinciden con el total que declara la página. O sea: si un mes paginara, el notebook se detiene en vez de entregar datos incompletos en silencio. Para una temporada con más normas por mes habría que recorrer las páginas siguientes.

2. **Solo leemos el título, no el cuerpo del decreto.** Los distritos y provincias exactos están en los anexos del PDF. Un departamento afectado pero no nombrado en el título no se cuenta.

3. **El conteo es por decreto, no por territorio afectado.** Un decreto que cubre 40 distritos de Piura pesa lo mismo que uno que cubre 1. Para comparar magnitud haría falta contar distritos, que es justamente una de las mejoras que la consigna sugiere como innovación.

4. **La búsqueda depende del término `"estado de emergencia precipitaciones"`.** Un decreto de lluvias redactado con otro vocabulario podría no aparecer en los resultados.

5. **Los selectores CSS son del HTML actual de gob.pe.** Si el sitio cambia su maquetado, la extracción deja de encontrar los datos. La tabla de verificación es la que avisaría.